# Project 2

Current regression and classification workflows supporting the report. Original course material remains unchanged in `source/`. The numerical core is in `reused_network.py` and `reused_project1.py`.


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

## Project packages

All core packages are imported here for later use. Jupyter is provided by JupyterLab. `jaxlib` is JAX's compiled backend. The brief treats autograd as an alternative to JAX, and PyTorch/TensorFlow as optional comparison tools.

Package choices follow the assignment brief; references are listed in the report bibliography.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — project setup, reused methods, checks or experiments.

</div>

> LLM-assisted: Codex implemented package imports and setup on 27 October 2026.


In [1]:
import numpy as np
import scipy as sp
import jupyter
import sklearn
import pandas as pd
import jax
import jax.numpy as jnp
import jaxlib


<div class="proj2-formal" style="color: #A6ADB8; background-color: #252733; border-left: 3px solid #A6ADB8; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

<span style="color: #A6ADB8; font-size: 0.75em; font-weight: 700; letter-spacing: 0.08em;">PROVENANCE / WORKFLOW NOTE</span>

## Reused prior work and scope

The implementation in `reused_project1.py` and `reused_network.py` reuses Project 1 and Weeks 41-44. Earlier work is credited in the code and report bibliography.

See the [project report](../report/main.pdf). From the project root, run `python Code/check_part_c.py` for activations, initialization and regression gradients; `python Code/check_part_d.py` for regularized objectives and gradients; and `python Code/check_part_e.py` for classification gradients and stopping behavior.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

### Answers

The covered MSE/penalty, sigmoid/ReLU and backpropagation material is assembled in the report. Part c adds Leaky ReLU and its derivative. Source: Weeks 42/43 (`priorWeek42`, `priorWeek43`).

**Covered prior work:** MSE/L1/L2 maths and term explanations, sigmoid/ReLU derivatives, and the Week 42 chain-rule backpropagation steps already exist. These are assembled in the report theory section with source references. They are reused theory, not newly developed methods. The joined network and saved small checks also exist.

<div class="proj2-formal" style="color: #A6ADB8; background-color: #252733; border-left: 3px solid #A6ADB8; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

<p>Part c now includes Leaky ReLU, its derivative and zero convention in the shared code and report. All three hidden activations passed independent JAX gradient checks.</p>

</div>

**Assembled from prior work:** Project 1 supplies MSE and the penalty gradients/subgradients. Week 42 cells 2/18/21 supply MSE and activation derivatives, and cells 15/26/47/54/59 supply chain-rule explanations and manual backpropagation.

For actual batch size $m$ and $K$ outputs,
$$C_{\mathrm{MSE}}=\frac{1}{mK}\sum_{i,k}(\hat y_{ik}-y_{ik})^2,
\qquad \frac{\partial C}{\partial\hat Y}=\frac{2}{mK}(\hat Y-Y).$$
The prior penalty gradients are $2\lambda\theta$ for L2 and $\lambda s$ for L1, where $s_j=\operatorname{sign}(\theta_j)$ away from zero and $s_j\in[-1,1]$ at zero. Project 1's JAX convention is +1 at zero. The supplied Week 44 course example uses 0. Part d now implements weight-only L1/L2 penalties with the L1 subgradient sign(0)=0. Biases are not penalized.

The existing activation expressions are $\sigma(z)=1/(1+e^{-z})$, $\sigma'(z)=\sigma(z)(1-\sigma(z))$, and $r(z)=\max(0,z)$ with derivative 1 for $z>0$ and 0 for $z<0$. The code assigns 0 at the ReLU kink.

For row-batch manual backpropagation, the earlier code uses
$$\delta^l=(\delta^{l+1}(W^{l+1})^T)\odot f_l'(z^l),\quad
\nabla_{W^l}C=(a^{l-1})^T\delta^l,\quad
\nabla_{b^l}C=\sum_i\delta^l_{i,:}.$$
These are assembled from the earlier work, not new backpropagation contributions by Codex. See the report for the complete notation and source credit. The Week 44 supplied penalty example is course material, not a completed student solution.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — project setup, reused methods, checks or experiments.

</div>

> LLM-assisted: Codex integrated the imported coursework functions on 27 October 2026; source credits remain in reused_network.py.


In [ ]:
# Reused methods: Week 42 cells 2/18/21/59/63 and Week 43 cell 14.
# Implementations and adaptation comments are in reused_network.py.
from reused_network import (
    activation, activation_derivative, binary_cross_entropy,
    network_loss, backpropagation_batched,
)


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

### Answers

This Runge study reuses `reused_project1.py` for the data, scaling, OLS and optimizer updates, and `reused_network.py` for the forward pass, manual backpropagation and training loop. Sources: `priorProject1`, `priorWeek41`, `priorWeek42`, `priorWeek43`, and `priorWeek44` in the report bibliography.

**Fixed settings:** float64, sigmoid hidden layers, linear output, full MSE with no regularization, zero biases, 1000 epochs, seed 2026 for initialization and batch shuffling. Samples are rows. Each layer has $W^{(l)}\in\mathbb R^{n_{\mathrm{in}}\times n_{\mathrm{out}}}$ and computes $z=aW+b$. Inputs are raw $x$ reshaped to one column, then scaled with training statistics. Targets are centered with their training mean and predictions are shifted back before scoring.

Weights use **Xavier/Glorot normal**, separately for each layer:
$$W_{ij}^{(l)}\sim\mathcal N\!\left(0,\frac{2}{n_{\mathrm{in}}+n_{\mathrm{out}}}\right),\qquad b^{(l)}=0.$$
The second normal parameter is **variance**. NumPy takes **standard deviation**, so `scale=np.sqrt(2.0 / (n_in + n_out))`. Both widths refer to the individual layer, including the output layer. Source: [course textbook §8.10, Eq. (8.73)](https://educationalmaterialuio.github.io/MachineLearningUiO/doc/LectureNotes/_build/html/chapter8.html#weight-initialisation), checked by Codex on 2 November 2026. The shared initializer keeps its older default for earlier callers. This study passes `initialization="xavier"` explicitly.

The two-hidden-layer network has **100 nodes in each hidden layer, 200 hidden nodes in total**.

**Searched settings:** `[1, 50, 1]` and `[1, 100, 100, 1]`, each with plain full-batch GD, minibatch SGD–RMSprop and minibatch SGD–Adam, at constant base learning rates 0.001, 0.01, 0.03 and 0.1. Adaptive optimizers still change their per-parameter updates. Their reused defaults are RMSprop $\rho=0.9$, Adam $\beta_1=0.9$, $\beta_2=0.999$, and $\epsilon=10^{-8}$.

**Data and pick rule:** reconstruct Project 1’s 1000 evenly spaced inputs, noise SD 0.1, data seed 42 and 80/20 train/test split with seed 42. Use **five-fold shuffled K-fold validation on the 800 training rows**, with fold seed 42 and the same folds for all settings and both OLS degrees. Each fold has 640 training and 160 validation rows. Fit feature scaling and target centering only on each fold’s training rows. Pick learning rates by **mean validation MSE across all five folds**, within each architecture/optimizer pair. Mark the overall network CV pick before any test scoring. Refit those six fixed picks and both fixed OLS degrees on all 800 training rows with freshly fitted training-only scaling, then score the 200 test rows. No test-based tuning follows.

**Fold spread:** report the sample SD of the five validation MSEs alongside their mean. This is variation across these folds, not uncertainty over initialization seeds or a confidence interval. Initialization and shuffling stay fixed at seed 2026. All runs use a fixed 1000-epoch budget without an asserted convergence threshold.

**Benchmark provenance:** degree 14 was Project 1’s five-fold CV pick. Degree 15 was its historical test-sweep minimum, shown separately to meet the requested best-degree comparison. Both degrees are fixed before this search. The old folds used all 1000 points and the old degree sweep scored the test split, so these reused test rows are not a previously unseen holdout. The new network search does not read test scores until its picks are fixed. Earlier holdout scores remain historical in `../results/runge_part_b/`. The current five-fold study is saved separately in `../results/runge_part_b_cv/`.

<div class="proj2-formal" style="color: #A6ADB8; background-color: #252733; border-left: 3px solid #A6ADB8; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

> **K-fold update:** At the student’s request, Codex replaced single-split validation with shared five-fold validation on 2 November 2026. Network and optimizer implementations are reused unchanged. Earlier credits and outputs are kept.

</div>

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### Runge data, shared folds and training-only scaling

This study saves the exact outer split, fold indices and fold-specific scaling statistics.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — project setup, reused methods, checks or experiments.

</div>

> Student: Specified the Runge data, split, shared folds and training-only scaling. LLM-assisted: Codex implemented setup, preprocessing, data records and persistence on 2 November 2026.


In [2]:
from pathlib import Path
import json
import time
import importlib.metadata
from IPython.display import display
from sklearn.model_selection import train_test_split
from reused_project1 import (
    generate_data, scale_data, create_design_matrix, fit_regression, predict,
    make_cv_splits,
)
from reused_network import (
    create_layers_batched, feed_forward_batched, network_loss,
    backpropagation_batched, train_network,
)

runge_out = Path.cwd() / "../results/runge_part_b_cv"
if not (Path.cwd() / "reused_network.py").exists():
    runge_out = Path("results/runge_part_b_cv")
runge_out = runge_out.resolve()
runge_out.mkdir(parents=True, exist_ok=True)
jax.config.update("jax_enable_x64", True)
runge_x, runge_y, runge_truth = generate_data(1000, noise_std=0.1, seed=42)
runge_train_pool, runge_test = train_test_split(
    np.arange(len(runge_x)), test_size=0.2, random_state=42,
)
# Recycled: Project 1 make_cv_splits uses shuffled sklearn KFold.
runge_folds = make_cv_splits(len(runge_train_pool), k=5, seed=42)
runge_fold_data = []
runge_fold_indices = {}
runge_fold_statistics = {}
for fold, (fit_position, val_position) in enumerate(runge_folds):
    fit_rows = runge_train_pool[fit_position]
    val_rows = runge_train_pool[val_position]
    X_fit, X_val, y_fit, y_val, x_mean, x_sd, y_mean = scale_data(
        runge_x[fit_rows, None], runge_x[val_rows, None],
        runge_y[fit_rows], runge_y[val_rows],
    )
    runge_fold_data.append({
        "X_fit": X_fit, "X_val": X_val,
        "y_fit": y_fit.reshape(-1, 1), "y_val": y_val.reshape(-1, 1),
        "y_mean": y_mean,
    })
    runge_fold_indices[f"fit_{fold}"] = fit_rows
    runge_fold_indices[f"validation_{fold}"] = val_rows
    runge_fold_statistics[f"x_mean_{fold}"] = x_mean
    runge_fold_statistics[f"x_sd_{fold}"] = x_sd
    runge_fold_statistics[f"y_mean_{fold}"] = y_mean
    assert not set(fit_rows) & set(val_rows)
    assert not set(fit_rows) & set(runge_test)
    assert not set(val_rows) & set(runge_test)
    np.testing.assert_allclose(X_fit.mean(), 0, atol=1e-14)
    np.testing.assert_allclose(X_fit.std(), 1, atol=1e-14)
np.testing.assert_array_equal(
    np.sort(np.concatenate([v for k, v in runge_fold_indices.items() if k.startswith("validation")])),
    np.sort(runge_train_pool),
)
np.savez(
    runge_out / "data_split.npz", x=runge_x, y=runge_y, truth=runge_truth,
    train_pool=runge_train_pool, test=runge_test, **runge_fold_indices,
)
np.savez(runge_out / "fold_preprocessing.npz", **runge_fold_statistics)
# The first fold also supplies a small development-check batch.
runge_X_fit = runge_fold_data[0]["X_fit"]
runge_y_fit = runge_fold_data[0]["y_fit"]
print("Outer training/test: 800/200. Five shared folds: 640 fit / 160 validation rows each.")


Outer training/test: 800/200. Five shared folds: 640 fit / 160 validation rows each.


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### Development checks: every weight and bias

Run this cell during development, outside the experiment loop. JAX checks every gradient entry for both full requested architectures, at initialization and after a small perturbation. The initializer is also checked against seeded NumPy draws, layer by layer.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — project setup, reused methods, checks or experiments.

</div>

> LLM-assisted: Codex designed and implemented the independent JAX gradient, initialization and optimizer checks on 2 November 2026.


In [3]:
def runge_jax_loss(layers, inputs, targets):
    """Evaluate an independent JAX MSE objective for the gradient checks.
    
    LLM-assisted: Codex implemented this scalar reference objective.
    """
    a = jnp.asarray(inputs)
    for W, b in layers[:-1]:
        a = jax.nn.sigmoid(a @ W + b)
    W, b = layers[-1]
    return jnp.mean((a @ W + b - targets) ** 2)

runge_gradient_records = []
for widths in [(1, 50, 1), (1, 100, 100, 1)]:
    initial = create_layers_batched(
        widths[0], widths[1:], seed=2026, initialization="xavier",
    )
    expected_rng = np.random.default_rng(2026)
    for (W, b), n_in, n_out in zip(initial, widths[:-1], widths[1:]):
        expected = expected_rng.normal(
            loc=0.0, scale=np.sqrt(2.0 / (n_in + n_out)),
            size=(n_in, n_out),
        )
        np.testing.assert_array_equal(W, expected)
        np.testing.assert_array_equal(b, np.zeros(n_out))
    check_rng = np.random.default_rng(17)
    perturbed = [
        (W + check_rng.normal(0, 0.02, W.shape),
         b + check_rng.normal(0, 0.02, b.shape))
        for W, b in initial
    ]
    names = ["sigmoid"] * (len(widths) - 2) + ["linear"]
    for point, layers in [("initial", initial), ("perturbed", perturbed)]:
        manual = backpropagation_batched(
            runge_X_fit[:7], layers, names, runge_y_fit[:7],
        )
        automatic = jax.grad(runge_jax_loss)(
            layers, runge_X_fit[:7], runge_y_fit[:7],
        )
        for layer, (manual_pair, auto_pair) in enumerate(zip(manual, automatic)):
            for parameter, actual, expected in zip(["W", "b"], manual_pair, auto_pair):
                np.testing.assert_allclose(actual, expected, atol=1e-11, rtol=1e-9)
                runge_gradient_records.append({
                    "architecture": str(list(widths)), "point": point,
                    "layer": layer, "parameter": parameter,
                    "entries": actual.size,
                    "max_abs_error": float(np.max(np.abs(actual - expected))),
                })
runge_gradient_table = pd.DataFrame(runge_gradient_records)
runge_gradient_table.to_csv(runge_out / "gradient_checks.csv", index=False)
display(runge_gradient_table)

# Check each optimizer on a small run before the full search.
for method in ["gd", "rmsprop", "adam"]:
    pilot_initial = create_layers_batched(1, [50, 1], initialization="xavier")
    _, pilot_loss = train_network(
        pilot_initial, runge_X_fit[:32], runge_y_fit[:32],
        ["sigmoid", "linear"], method=method, gamma=0.001,
        epochs=3, batch_size=16 if method != "gd" else 32,
    )
    assert np.isfinite(pilot_loss).all()
print("All weight/bias checks and three finite training pilots passed.")

,architecture,point,layer,parameter,entries,max_abs_error
0,"[1, 50, 1]",initial,0,W,50,3.469447e-18
1,"[1, 50, 1]",initial,0,b,50,4.163336e-17
2,"[1, 50, 1]",initial,1,W,50,2.220446e-16
3,"[1, 50, 1]",initial,1,b,1,4.440892e-16
4,"[1, 50, 1]",perturbed,0,W,50,1.387779e-17
5,"[1, 50, 1]",perturbed,0,b,50,2.775558e-17
6,"[1, 50, 1]",perturbed,1,W,50,2.220446e-16
7,"[1, 50, 1]",perturbed,1,b,1,2.220446e-16
8,"[1, 100, 100, 1]",initial,0,W,100,3.035766e-18
9,"[1, 100, 100, 1]",initial,0,b,100,1.084202e-18


All weight/bias checks and three finite training pilots passed.


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### Reused training loop

Each batch follows forward pass → MSE loss → manual backpropagation → Project 1 optimizer update. The shared trainer checks epoch loss for nonfinite values. GD uses all fit rows, while RMSprop and Adam shuffle minibatches of 32, including the smaller final batch. The code below shows the actual reused trainer.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### Five-fold grid search

All 24 settings use the same five folds, initialization seed and 1000-epoch budget. This gives 120 fits. Each fold fits its own preprocessing on its 640 training rows. Learning rates stay fixed. Epoch-limit completion does not assert convergence. Times cover training and fold scoring, excluding development checks. Failed folds stay visible and exclude that setting from model picks.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — project setup, reused methods, checks or experiments.

</div>

> Student: Specified the architectures, optimizer/learning-rate grid, shared five-fold design and training budget. LLM-assisted: Codex implemented the experiment loop, timing, failure handling and output files on 2 November 2026.


In [5]:
runge_architectures = [(1, 50, 1), (1, 100, 100, 1)]
runge_rates = [0.001, 0.01, 0.03, 0.1]
runge_epochs = 1000
runge_records = []
runge_fold_records = []
runge_histories = {}
runge_oof_predictions = {}
for widths in runge_architectures:
    initial = create_layers_batched(1, widths[1:], initialization="xavier", seed=2026)
    names = ["sigmoid"] * (len(widths) - 2) + ["linear"]
    for method in ["gd", "rmsprop", "adam"]:
        for gamma in runge_rates:
            run_id = f"run_{len(runge_records):02d}"
            setting_folds = []
            oof = np.full(len(runge_train_pool), np.nan)
            for fold, ((fit_position, val_position), data) in enumerate(zip(runge_folds, runge_fold_data)):
                batch_size = len(fit_position) if method == "gd" else 32
                start = time.perf_counter()
                row = {
                    "run_id": run_id, "architecture": str(list(widths)),
                    "optimizer": method, "gamma": gamma, "fold": fold,
                    "epochs": runge_epochs, "batch_size": batch_size,
                    "seed": 2026, "status": "fixed_epoch_limit", "error": "",
                }
                try:
                    with np.errstate(over="raise", invalid="raise", divide="raise"):
                        layers, history = train_network(
                            initial, data["X_fit"], data["y_fit"], names,
                            gamma=gamma, epochs=runge_epochs, batch_size=batch_size,
                            method=method, seed=2026,
                        )
                        val_prediction = feed_forward_batched(data["X_val"], layers, names)
                        validation_mse = float(np.mean((val_prediction - data["y_val"]) ** 2))
                    row.update(train_mse=float(history[-1]), validation_mse=validation_mse)
                    oof[val_position] = val_prediction.ravel() + data["y_mean"]
                    runge_histories[f"{run_id}_fold_{fold}"] = history
                    np.savez(
                        runge_out / f"{run_id}_fold_{fold}_weights.npz",
                        **{f"W_{i}": W for i, (W, b) in enumerate(layers)},
                        **{f"b_{i}": b for i, (W, b) in enumerate(layers)},
                    )
                except FloatingPointError as error:
                    # keep failed folds visible and out of model picks.
                    row.update(status="failed", error=str(error), train_mse=np.nan, validation_mse=np.nan)
                row["seconds"] = time.perf_counter() - start
                setting_folds.append(row)
                runge_fold_records.append(row)
            fold_table = pd.DataFrame(setting_folds)
            complete = (fold_table["status"] != "failed").all()
            runge_records.append({
                "run_id": run_id, "architecture": str(list(widths)),
                "optimizer": method, "gamma": gamma, "epochs": runge_epochs,
                "seed": 2026, "folds": 5,
                "status": "fixed_epoch_limit" if complete else "failed",
                "train_mse": fold_table["train_mse"].mean() if complete else np.nan,
                "cv_mse": fold_table["validation_mse"].mean() if complete else np.nan,
                "cv_sd": fold_table["validation_mse"].std(ddof=1) if complete else np.nan,
                "cv_seconds": fold_table["seconds"].sum(),
            })
            runge_oof_predictions[run_id] = oof
            pd.DataFrame(runge_fold_records).to_csv(runge_out / "fold_scores.csv", index=False)
            pd.DataFrame(runge_records).to_csv(runge_out / "grid.csv", index=False)
            print(run_id, list(widths), method, gamma, "five folds complete", flush=True)
runge_grid = pd.DataFrame(runge_records)
np.savez(runge_out / "training_curves.npz", **runge_histories)
np.savez(runge_out / "oof_predictions.npz", **runge_oof_predictions)
display(runge_grid)


run_00 [1, 50, 1] gd 0.001 five folds complete


run_01 [1, 50, 1] gd 0.01 five folds complete


run_02 [1, 50, 1] gd 0.03 five folds complete


run_03 [1, 50, 1] gd 0.1 five folds complete


run_04 [1, 50, 1] rmsprop 0.001 five folds complete


run_05 [1, 50, 1] rmsprop 0.01 five folds complete


... (15 outputs left out)

run_21 [1, 100, 100, 1] adam 0.01 five folds complete


run_22 [1, 100, 100, 1] adam 0.03 five folds complete


run_23 [1, 100, 100, 1] adam 0.1 five folds complete


,run_id,architecture,optimizer,gamma,epochs,seed,folds,status,train_mse,cv_mse,cv_sd,cv_seconds
0,run_00,"[1, 50, 1]",gd,0.001,1000,2026,5,fixed_epoch_limit,0.093980,0.094381,0.011810,6.129042
1,run_01,"[1, 50, 1]",gd,0.010,1000,2026,5,fixed_epoch_limit,0.091936,0.092486,0.010856,7.040474
2,run_02,"[1, 50, 1]",gd,0.030,1000,2026,5,fixed_epoch_limit,0.091867,0.092420,0.010837,6.262841
3,run_03,"[1, 50, 1]",gd,0.100,1000,2026,5,fixed_epoch_limit,0.078119,0.078699,0.011378,5.702053
4,run_04,"[1, 50, 1]",rmsprop,0.001,1000,2026,5,fixed_epoch_limit,0.010097,0.010417,0.000940,14.672893
5,run_05,"[1, 50, 1]",rmsprop,0.010,1000,2026,5,fixed_epoch_limit,0.009920,0.010445,0.001034,13.685812
6,run_06,"[1, 50, 1]",rmsprop,0.030,1000,2026,5,fixed_epoch_limit,0.010542,0.011110,0.001123,14.106210
7,run_07,"[1, 50, 1]",rmsprop,0.100,1000,2026,5,fixed_epoch_limit,0.013841,0.014306,0.003781,13.721852
8,run_08,"[1, 50, 1]",adam,0.001,1000,2026,5,fixed_epoch_limit,0.009829,0.009979,0.000768,15.587377
9,run_09,"[1, 50, 1]",adam,0.010,1000,2026,5,fixed_epoch_limit,0.009677,0.010026,0.000849,15.674739


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### CV picks, OLS folds and learning-rate plots

Pick a learning rate within each architecture/optimizer pair by mean MSE over all five validation folds. Mark the overall network CV pick before test scoring. Both fixed Project 1 OLS degrees use the same folds and training-fold-only polynomial scaling. The fold SD is descriptive spread across splits, not a confidence interval or seed uncertainty. The learning curves show mean training MSE across folds for each picked setting.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — project setup, reused methods, checks or experiments.

</div>

> Student: Specified selection by mean validation MSE and matched-fold OLS comparisons. LLM-assisted: Codex implemented selection, aggregation, tables and plots on 2 November 2026.


In [ ]:
runge_success = runge_grid.loc[runge_grid["status"] != "failed"]
runge_picks = runge_success.loc[
    runge_success.groupby(["architecture", "optimizer"])["cv_mse"].idxmin()
].copy()
if len(runge_picks) != 6:
    raise RuntimeError("At least one architecture/optimizer pair has no complete CV setting")
runge_best_id = runge_picks.loc[runge_picks["cv_mse"].idxmin(), "run_id"]
runge_picks["overall_validation_pick"] = runge_picks["run_id"] == runge_best_id
runge_picks.to_csv(runge_out / "validation_picks.csv", index=False)
display(runge_picks)

runge_ols_folds = []
runge_ols_cv = []
runge_ols_oof = {}
for degree, provenance in [(14, "prior CV pick"), (15, "historical test-sweep pick")]:
    oof = np.full(len(runge_train_pool), np.nan)
    for fold, (fit_position, val_position) in enumerate(runge_folds):
        start = time.perf_counter()
        fit_rows = runge_train_pool[fit_position]
        val_rows = runge_train_pool[val_position]
        X_fit = create_design_matrix(runge_x[fit_rows], degree)
        X_val = create_design_matrix(runge_x[val_rows], degree)
        X_fit, X_val, y_fit, _, _, _, y_mean = scale_data(X_fit, X_val, runge_y[fit_rows])
        theta = fit_regression(X_fit, y_fit, lambda_=0.0)
        pred_val = predict(X_val, theta, y_mean)
        pred_fit = predict(X_fit, theta, y_mean)
        oof[val_position] = pred_val
        runge_ols_folds.append({
            "degree": degree, "fold": fold,
            "train_mse": float(np.mean((runge_y[fit_rows] - pred_fit) ** 2)),
            "validation_mse": float(np.mean((runge_y[val_rows] - pred_val) ** 2)),
            "seconds": time.perf_counter() - start,
        })
    table = pd.DataFrame(runge_ols_folds)
    table = table.loc[table["degree"] == degree]
    runge_ols_cv.append({
        "degree": degree, "provenance": provenance,
        "train_mse": table["train_mse"].mean(),
        "cv_mse": table["validation_mse"].mean(),
        "cv_sd": table["validation_mse"].std(ddof=1),
        "cv_seconds": table["seconds"].sum(),
    })
    runge_ols_oof[f"ols_degree_{degree}"] = oof
runge_ols_cv = pd.DataFrame(runge_ols_cv)
pd.DataFrame(runge_ols_folds).to_csv(runge_out / "ols_fold_scores.csv", index=False)
runge_ols_cv.to_csv(runge_out / "ols_cv.csv", index=False)
np.savez(runge_out / "ols_oof_predictions.npz", **runge_ols_oof)
display(runge_ols_cv)



<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### Refit fixed CV picks, then score the reused test split

Refit the six CV-picked network settings and both fixed Project 1 OLS degrees on all 800 outer training rows. Fit new preprocessing only on those rows. Keep CV MSE and fold SD separate from final refit training MSE and test MSE. Save CV search time and final fitting time separately. Earlier test scores do not change the fixed grid, epoch budget or pick rule. This remains the historically used Project 1 test split.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — project setup, reused methods, checks or experiments.

</div>

> Student: Specified full-development refits and evaluation of the fixed picks on the reused test split. LLM-assisted: Codex implemented refitting, scoring, plots and output files on 2 November 2026.


In [ ]:
runge_X_train, runge_X_test, runge_y_train, _, runge_x_mean, runge_x_sd, runge_y_mean = scale_data(
    runge_x[runge_train_pool, None], runge_x[runge_test, None], runge_y[runge_train_pool],
)
runge_y_train = runge_y_train.reshape(-1, 1)
np.savez(runge_out / "refit_preprocessing.npz", x_mean=runge_x_mean, x_sd=runge_x_sd, y_mean=runge_y_mean)
runge_comparison = []
runge_test_predictions = {}
runge_refit_histories = {}
for _, row in runge_picks.iterrows():
    widths = json.loads(row.architecture)
    names = ["sigmoid"] * (len(widths) - 2) + ["linear"]
    initial = create_layers_batched(1, widths[1:], initialization="xavier", seed=2026)
    batch_size = len(runge_train_pool) if row.optimizer == "gd" else 32
    start = time.perf_counter()
    layers, history = train_network(
        initial, runge_X_train, runge_y_train, names,
        gamma=row.gamma, epochs=runge_epochs, batch_size=batch_size,
        method=row.optimizer, seed=2026,
    )
    fit_seconds = time.perf_counter() - start
    prediction = feed_forward_batched(runge_X_test, layers, names).ravel() + runge_y_mean
    runge_test_predictions[row.run_id] = prediction
    runge_refit_histories[row.run_id] = history
    np.savez(
        runge_out / f"{row.run_id}_refit_weights.npz",
        **{f"W_{i}": W for i, (W, b) in enumerate(layers)},
        **{f"b_{i}": b for i, (W, b) in enumerate(layers)},
    )
    runge_comparison.append({
        "model": row.architecture + " " + row.optimizer,
        "run_id": row.run_id, "gamma": row.gamma,
        "cv_mse": row.cv_mse, "cv_sd": row.cv_sd,
        "refit_train_mse": float(history[-1]),
        "test_mse": float(np.mean((runge_y[runge_test] - prediction) ** 2)),
        "cv_seconds": row.cv_seconds, "refit_seconds": fit_seconds,
        "overall_validation_pick": row.overall_validation_pick,
    })
for _, row in runge_ols_cv.iterrows():
    degree = int(row.degree)
    start = time.perf_counter()
    X_train = create_design_matrix(runge_x[runge_train_pool], degree)
    X_test = create_design_matrix(runge_x[runge_test], degree)
    X_train, X_test, y_train, _, mean_x, sd_x, mean_y = scale_data(
        X_train, X_test, runge_y[runge_train_pool],
    )
    theta = fit_regression(X_train, y_train, lambda_=0.0)
    fit_seconds = time.perf_counter() - start
    pred_train = predict(X_train, theta, mean_y)
    pred_test = predict(X_test, theta, mean_y)
    run_id = f"ols_degree_{degree}"
    runge_test_predictions[run_id] = pred_test
    runge_comparison.append({
        "model": f"OLS degree {degree} ({row.provenance})", "run_id": run_id,
        "gamma": np.nan, "cv_mse": row.cv_mse, "cv_sd": row.cv_sd,
        "refit_train_mse": float(np.mean((runge_y[runge_train_pool] - pred_train) ** 2)),
        "test_mse": float(np.mean((runge_y[runge_test] - pred_test) ** 2)),
        "cv_seconds": row.cv_seconds, "refit_seconds": fit_seconds,
        "overall_validation_pick": False,
    })
    np.savez(runge_out / f"{run_id}_refit_weights.npz", theta=theta, x_mean=mean_x, x_sd=sd_x, y_mean=mean_y)
runge_comparison = pd.DataFrame(runge_comparison)
runge_comparison.to_csv(runge_out / "test_comparison.csv", index=False)
np.savez(runge_out / "test_predictions.npz", **runge_test_predictions)
np.savez(runge_out / "refit_training_curves.npz", **runge_refit_histories)
display(runge_comparison)
print("Overall network CV pick:", runge_best_id)
print("Fold SD describes split variation at fixed seeds, not a confidence interval.")
print("The Project 1 test split has been used before. No test-based tuning follows.")
runge_settings = {
    "date": "2026-11-02", "architectures": runge_architectures,
    "rates": runge_rates, "epochs": runge_epochs,
    "data_seed": 42, "outer_split_seed": 42, "fold_seed": 42, "folds": 5,
    "initialization_seed": 2026, "shuffle_seed": 2026,
    "initialization": "xavier_normal", "dtype": "float64",
    "loss": "mean over all output entries, no penalty",
    "scaling": "fit separately on each fold training set, refitted on 800 outer training rows for final fits",
    "pick_rule": "lowest mean validation MSE over all five folds per architecture/optimizer pair",
    "fold_spread": "sample SD, ddof=1, not seed uncertainty or a confidence interval",
    "overall_validation_pick": runge_best_id,
    "test_provenance": "reused Project 1 split, previously scored in degree sweep, old CV and earlier holdout runs",
    "convergence": "fixed epoch limit, no convergence assertion or early stopping",
    "dependencies": {package: importlib.metadata.version(package) for package in [
        "numpy", "scipy", "matplotlib", "pandas", "seaborn", "scikit-learn", "jax", "jaxlib", "nbclient", "nbformat",
    ]},
    "rerun": "Run package imports and Part b cells in order with the project .venv kernel, working directory Code/",
}
(runge_out / "settings.json").write_text(json.dumps(runge_settings, indent=2) + "\n")


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

### Answers

The Part c comparison keeps the Runge data, outer split, shared five folds and training-only scaling from Part b. It uses **Adam**, the optimizer of the overall Part b CV pick, for every setting. The output stays **linear**, with full **MSE and no regularization**. Training uses the faster shared NumPy implementation, 1000 epochs, minibatches of 32, initialization/shuffle seed 2026 and fold seed 42. These are fixed settings, not new tuned picks.

**Compared settings:** hidden activations `sigmoid`, `relu` and `leaky_relu`, hidden depths 1 and 2, and widths 25, 50 and 100 per hidden layer. The six hidden-width lists are `[25]`, `[50]`, `[100]`, `[25, 25]`, `[50, 50]` and `[100, 100]`. Each activation/architecture pair rechecks constant base learning rates **0.0003, 0.001 and 0.003**. This gives 54 settings and 270 fold fits. Exact matches from Part b are reused with their source and original time stated.

**Initialization:** sigmoid hidden layers use Xavier normal, with variance $2/(n_{\mathrm{in}}+n_{\mathrm{out}})$. ReLU and Leaky ReLU hidden layers use **He normal**, with variance $2/n_{\mathrm{in}}$, following [course §8.10, Eq. (8.74)](https://educationalmaterialuio.github.io/MachineLearningUiO/doc/LectureNotes/_build/html/chapter8.html#weight-initialisation). NumPy receives the square root as its standard deviation. Every linear output layer uses Xavier normal and all biases start at zero. The initializer accepts a separate scheme per layer. Because initialization is matched to activation, these are comparisons of the stated activation/initialization pairs, not an isolated activation-only effect.

**Leaky ReLU:** the negative slope is fixed at $\alpha=0.01$:
$$f(z)=\begin{cases}z,&z>0\\\alpha z,&z\leq0,\end{cases}\qquad
f'(z)=\begin{cases}1,&z>0\\\alpha,&z<0.\end{cases}$$
At zero the mathematical derivative is undefined. The code assigns $\alpha$ there, matching its negative branch. ReLU keeps its existing zero derivative at zero. Activation names select both the forward function and its matching manual backward derivative.

**Pick rule:** pick the learning rate separately for each activation/architecture pair by lowest mean validation MSE over the same five folds. Show mean training MSE, mean validation MSE, sample fold SD and the paired validation-minus-training gap. A larger gap together with worse validation error as capacity grows is evidence consistent with overfitting. A positive gap alone does not establish it. This use of held-out error follows [Nielsen, Chapter 3, “Overfitting and regularization”](http://neuralnetworksanddeeplearning.com/chap3.html#overfitting_and_regularization).

The fold SD describes split variation at fixed seeds, not seed uncertainty or a confidence interval. These CV folds also pick learning rates, so their means are tuning evidence rather than an independent final assessment. Part c does **not** score or tune against the outer test set. Reaching 1000 epochs is not an assertion of convergence. The optional bias–variance decomposition is outside this requested comparison.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### Shared data, folds and fixed settings

Read the saved Part b data and fold membership. Every fold fits its own feature scaling and target mean. No Part b training or test assessment is rerun here. Save the settings before starting the grid.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — Part c.

</div>

> Student: Specified the depth/width, activation, initialization and learning-rate comparisons. LLM-assisted: Codex implemented Part c configuration, setup and records on 2 November 2026.


In [1]:
from pathlib import Path
import json
import hashlib
import importlib.metadata
import shutil
import time
import runpy
import numpy as np
import pandas as pd
from IPython.display import display
from reused_project1 import scale_data
from reused_network import (
    LEAKY_RELU_SLOPE, create_layers_batched, train_network,
    feed_forward_batched,
)

c_code_dir = Path.cwd() if (Path.cwd() / "reused_network.py").exists() else Path.cwd() / "Code"
c_root = c_code_dir.parent
c_out = c_root / "results/runge_part_c"
c_out.mkdir(exist_ok=True)
c_b_out = c_root / "results/runge_part_b_cv"
c_b_settings = json.loads((c_b_out / "settings.json").read_text())
c_b_picks = pd.read_csv(c_b_out / "validation_picks.csv")
c_b_pick = c_b_picks.loc[c_b_picks["overall_validation_pick"]].iloc[0]
c_optimizer = c_b_pick.optimizer
c_depths = [1, 2]
c_widths = [25, 50, 100]
c_activations = ["sigmoid", "relu", "leaky_relu"]
c_rates = [0.0003, 0.001, 0.003]
c_epochs = 1000
c_seed = 2026
assert c_b_settings["epochs"] == c_epochs
assert c_b_settings["initialization_seed"] == c_seed
assert c_b_settings["shuffle_seed"] == c_seed
assert c_b_settings["fold_seed"] == 42 and c_b_settings["folds"] == 5
c_split = np.load(c_b_out / "data_split.npz")
c_x, c_y = c_split["x"], c_split["y"]
c_train, c_test = c_split["train_pool"], c_split["test"]
c_folds = []
c_fold_data = []
c_statistics = {}
for fold in range(5):
    fit_rows = c_split[f"fit_{fold}"]
    validation_rows = c_split[f"validation_{fold}"]
    assert not set(fit_rows) & set(validation_rows)
    assert not set(fit_rows) & set(c_test)
    assert not set(validation_rows) & set(c_test)
    X_fit, X_val, y_fit, y_val, x_mean, x_sd, y_mean = scale_data(
        c_x[fit_rows, None], c_x[validation_rows, None], c_y[fit_rows], c_y[validation_rows],
    )
    c_folds.append((fit_rows, validation_rows))
    c_fold_data.append({
        "X_fit": X_fit, "X_val": X_val,
        "y_fit": y_fit.reshape(-1, 1), "y_val": y_val.reshape(-1, 1), "y_mean": y_mean,
    })
    c_statistics[f"x_mean_{fold}"] = x_mean
    c_statistics[f"x_sd_{fold}"] = x_sd
    c_statistics[f"y_mean_{fold}"] = y_mean
np.savez(c_out / "data_split.npz", **{key: c_split[key] for key in c_split.files})
np.savez(c_out / "fold_preprocessing.npz", **c_statistics)
c_settings = {
    "date": "2026-11-02", "optimizer": c_optimizer,
    "optimizer_source": "Part b overall CV pick, validation_picks.csv",
    "part_b_pick": c_b_pick.to_dict(),
    "activations": c_activations, "depths": c_depths, "widths": c_widths,
    "rates": c_rates, "epochs": c_epochs, "batch_size": 32,
    "seed": c_seed, "fold_seed": 42, "folds": 5,
    "leaky_relu_slope": LEAKY_RELU_SLOPE,
    "relu_derivative_at_zero": 0.0, "leaky_relu_derivative_at_zero": LEAKY_RELU_SLOPE,
    "hidden_initialization": {"sigmoid": "xavier_normal", "relu": "he_normal", "leaky_relu": "he_normal"},
    "linear_output_initialization": "xavier_normal", "bias_initialization": "zero",
    "loss": "full unregularized MSE", "dtype": "float64",
    "pick_rule": "lowest mean validation MSE per activation/depth/width, all five folds required",
    "gap": "paired validation MSE minus training MSE, sample SD ddof=1",
    "fold_spread": "sample SD, not seed uncertainty or a confidence interval",
    "test_use": "outer test indices excluded, no test prediction or scoring in Part c",
    "convergence": "fixed epoch limit, no early stopping or convergence assertion",
    "implementation_sha256": hashlib.sha256((c_code_dir / "reused_network.py").read_bytes()).hexdigest(),
    "dependencies": {name: importlib.metadata.version(name) for name in ["numpy", "scipy", "pandas", "matplotlib", "seaborn", "scikit-learn", "jax", "jaxlib"]},
    "rerun": ".venv/bin/python Code/run_part_c.py from the project root",
}
(c_out / "settings.json").write_text(json.dumps(c_settings, indent=2) + "\n")
print("Optimizer kept from Part b:", c_optimizer)
print("54 settings, five shared folds, 640/160 training/validation rows per fold.")

Optimizer kept from Part b: adam
54 settings, five shared folds, 640/160 training/validation rows per fold.


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### Development checks

Inspect the saved independent JAX checks. They cover every weight and bias for all 18 activation/architecture pairs at initialization and after perturbation. Rerun the check script if the core implementation has changed. No automatic differentiation is used for training.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — Part c.

</div>

> LLM-assisted: Codex implemented check execution, source-hash matching and result displays on 2 November 2026.


In [2]:
c_check_file = c_out / "development_checks.json"
c_check = json.loads(c_check_file.read_text()) if c_check_file.exists() else {}
if c_check.get("implementation_sha256") != c_settings["implementation_sha256"]:
    runpy.run_path(str(c_code_dir / "check_part_c.py"), run_name="__main__")
    c_check = json.loads(c_check_file.read_text())
    c_check["implementation_sha256"] = c_settings["implementation_sha256"]
    c_check_file.write_text(json.dumps(c_check, indent=2) + "\n")
display(pd.Series(c_check, name="Part c development checks"))
display(pd.read_csv(c_out / "gradient_checks.csv"))

checks                         [all 18 activation/architecture combinations c...
gradient_arrays_checked                                                      180
gradient_entries_checked                                                   86136
max_gradient_absolute_error                                                  0.0
leaky_relu_slope                                                            0.01
implementation_sha256          56f8997a1c1245154dc1019804190f510d29c5c456d3c9...
Name: Part c development checks, dtype: object

,activation,depth,width,point,layer,parameter,entries,max_abs_error
0,sigmoid,1,25,initial,0,W,25,5.551115e-17
1,sigmoid,1,25,initial,0,b,25,5.551115e-17
2,sigmoid,1,25,initial,1,W,25,4.440892e-16
3,sigmoid,1,25,initial,1,b,1,0.000000e+00
4,sigmoid,1,25,perturbed,0,W,25,1.387779e-17
...,...,...,...,...,...,...,...,...
175,leaky_relu,2,100,perturbed,0,b,100,1.110223e-16
176,leaky_relu,2,100,perturbed,1,W,10000,2.220446e-16
177,leaky_relu,2,100,perturbed,1,b,100,5.551115e-17
178,leaky_relu,2,100,perturbed,2,W,100,8.881784e-16


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### Activation, depth, width and learning-rate search

Train each setting on the same five folds. Reuse only exact matching sigmoid/Adam Part b fits, with original scores, weights and curves. New fits use the faster shared trainer. A failed fold excludes its whole setting from picks, and the failure stays visible.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — Part c.

</div>

> Student: Specified the shared-fold capacity and activation experiment and reuse of matching Part b results. LLM-assisted: Codex implemented training orchestration, reuse matching, failure handling and saved outputs on 2 November 2026.


In [3]:
c_b_grid = pd.read_csv(c_b_out / "grid.csv")
c_b_folds = pd.read_csv(c_b_out / "fold_scores.csv")
c_b_curves = np.load(c_b_out / "training_curves.npz")
c_b_oof = np.load(c_b_out / "oof_predictions.npz")
c_records = []
c_fold_records = []
c_histories = {}
c_oof_predictions = {}
for hidden_activation in c_activations:
    for depth in c_depths:
        for width in c_widths:
            widths = [1] + [width] * depth + [1]
            names = [hidden_activation] * depth + ["linear"]
            schemes = ["xavier" if hidden_activation == "sigmoid" else "he"] * depth + ["xavier"]
            initial = create_layers_batched(1, widths[1:], seed=c_seed, initialization=schemes)
            for gamma in c_rates:
                run_id = f"c_{len(c_records):02d}"
                candidates = c_b_grid.loc[
                    (c_b_grid["architecture"] == str(widths)) &
                    (c_b_grid["optimizer"] == c_optimizer) &
                    (c_b_grid["gamma"] == gamma) &
                    (c_b_grid["status"] == "fixed_epoch_limit")
                ]
                reuse = hidden_activation == "sigmoid" and len(candidates) == 1
                source_id = candidates.iloc[0].run_id if reuse else ""
                oof = np.full(len(c_x), np.nan)
                setting_folds = []
                for fold, ((fit_rows, val_rows), data) in enumerate(zip(c_folds, c_fold_data)):
                    row = {
                        "run_id": run_id, "activation": hidden_activation,
                        "depth": depth, "width": width, "architecture": str(widths),
                        "optimizer": c_optimizer, "gamma": gamma, "fold": fold,
                        "epochs": c_epochs, "seed": c_seed,
                        "origin": "reused_part_b" if reuse else "new_part_c",
                        "source_run_id": source_id, "status": "fixed_epoch_limit", "error": "",
                    }
                    if reuse:
                        old = c_b_folds.loc[(c_b_folds["run_id"] == source_id) & (c_b_folds["fold"] == fold)].iloc[0]
                        row.update(train_mse=old.train_mse, validation_mse=old.validation_mse, seconds=old.seconds)
                        c_histories[f"{run_id}_fold_{fold}"] = c_b_curves[f"{source_id}_fold_{fold}"].copy()
                        shutil.copy2(c_b_out / f"{source_id}_fold_{fold}_weights.npz", c_out / f"{run_id}_fold_{fold}_weights.npz")
                        oof[c_train] = c_b_oof[source_id]
                    else:
                        start = time.perf_counter()
                        try:
                            with np.errstate(over="raise", invalid="raise", divide="raise"):
                                layers, history = train_network(
                                    initial, data["X_fit"], data["y_fit"], names,
                                    gamma=gamma, epochs=c_epochs, batch_size=32,
                                    method=c_optimizer, seed=c_seed,
                                )
                                val_prediction = feed_forward_batched(data["X_val"], layers, names)
                                validation_mse = float(np.mean((val_prediction - data["y_val"]) ** 2))
                                if not np.isfinite(validation_mse):
                                    raise FloatingPointError("Nonfinite final validation MSE")
                            row.update(train_mse=float(history[-1]), validation_mse=validation_mse)
                            oof[val_rows] = val_prediction.ravel() + data["y_mean"]
                            c_histories[f"{run_id}_fold_{fold}"] = history
                            np.savez(
                                c_out / f"{run_id}_fold_{fold}_weights.npz",
                                **{f"W_{i}": W for i, (W, b) in enumerate(layers)},
                                **{f"b_{i}": b for i, (W, b) in enumerate(layers)},
                            )
                        except FloatingPointError as error:
                            # retain failed settings instead of hiding them.
                            row.update(status="failed", error=str(error), train_mse=np.nan, validation_mse=np.nan)
                        row["seconds"] = time.perf_counter() - start
                    row["gap"] = row["validation_mse"] - row["train_mse"]
                    setting_folds.append(row)
                    c_fold_records.append(row)
                table = pd.DataFrame(setting_folds)
                complete = (table["status"] == "fixed_epoch_limit").all()
                c_records.append({
                    "run_id": run_id, "activation": hidden_activation,
                    "depth": depth, "width": width, "architecture": str(widths),
                    "optimizer": c_optimizer, "gamma": gamma,
                    "parameter_count": sum(W.size + b.size for W, b in initial),
                    "origin": "reused_part_b" if reuse else "new_part_c",
                    "source_run_id": source_id, "epochs": c_epochs, "seed": c_seed,
                    "status": "fixed_epoch_limit" if complete else "failed",
                    "train_mse": table["train_mse"].mean() if complete else np.nan,
                    "cv_mse": table["validation_mse"].mean() if complete else np.nan,
                    "cv_sd": table["validation_mse"].std(ddof=1) if complete else np.nan,
                    "gap": table["gap"].mean() if complete else np.nan,
                    "gap_sd": table["gap"].std(ddof=1) if complete else np.nan,
                    "seconds": table["seconds"].sum(),
                })
                c_oof_predictions[run_id] = oof[c_train]
                pd.DataFrame(c_fold_records).to_csv(c_out / "fold_scores.csv", index=False)
                pd.DataFrame(c_records).to_csv(c_out / "grid.csv", index=False)
                print(run_id, hidden_activation, widths, gamma, "reused" if reuse else "five new folds", flush=True)
c_grid = pd.DataFrame(c_records)
np.savez(c_out / "training_curves.npz", **c_histories)
np.savez(c_out / "oof_predictions.npz", **c_oof_predictions)
display(c_grid)

c_00 sigmoid [1, 25, 1] 0.0003 five new folds


c_01 sigmoid [1, 25, 1] 0.001 five new folds


c_02 sigmoid [1, 25, 1] 0.003 five new folds


c_03 sigmoid [1, 50, 1] 0.0003 five new folds


c_04 sigmoid [1, 50, 1] 0.001 reused


c_05 sigmoid [1, 50, 1] 0.003 five new folds


... (45 outputs left out)

c_51 leaky_relu [1, 100, 100, 1] 0.0003 five new folds


c_52 leaky_relu [1, 100, 100, 1] 0.001 five new folds


c_53 leaky_relu [1, 100, 100, 1] 0.003 five new folds


,run_id,activation,depth,width,architecture,optimizer,gamma,parameter_count,origin,source_run_id,epochs,seed,status,train_mse,cv_mse,cv_sd,gap,gap_sd,seconds
0,c_00,sigmoid,1,25,"[1, 25, 1]",adam,0.0003,76,new_part_c,,1000,2026,fixed_epoch_limit,0.049755,0.049759,0.004130,0.000004,0.008137,5.135387
1,c_01,sigmoid,1,25,"[1, 25, 1]",adam,0.0010,76,new_part_c,,1000,2026,fixed_epoch_limit,0.010226,0.010360,0.000600,0.000133,0.000793,5.122698
2,c_02,sigmoid,1,25,"[1, 25, 1]",adam,0.0030,76,new_part_c,,1000,2026,fixed_epoch_limit,0.009758,0.010007,0.000714,0.000250,0.000915,5.113097
3,c_03,sigmoid,1,50,"[1, 50, 1]",adam,0.0003,151,new_part_c,,1000,2026,fixed_epoch_limit,0.015216,0.015485,0.001241,0.000269,0.000726,7.263985
4,c_04,sigmoid,1,50,"[1, 50, 1]",adam,0.0010,151,reused_part_b,run_08,1000,2026,fixed_epoch_limit,0.009829,0.009979,0.000768,0.000150,0.000797,15.587377
5,c_05,sigmoid,1,50,"[1, 50, 1]",adam,0.0030,151,new_part_c,,1000,2026,fixed_epoch_limit,0.009739,0.010044,0.000928,0.000305,0.000993,7.276051
6,c_06,sigmoid,1,100,"[1, 100, 1]",adam,0.0003,301,new_part_c,,1000,2026,fixed_epoch_limit,0.016424,0.016694,0.001162,0.000270,0.001063,9.575738
7,c_07,sigmoid,1,100,"[1, 100, 1]",adam,0.0010,301,new_part_c,,1000,2026,fixed_epoch_limit,0.009808,0.010015,0.000662,0.000208,0.000781,9.417988
8,c_08,sigmoid,1,100,"[1, 100, 1]",adam,0.0030,301,new_part_c,,1000,2026,fixed_epoch_limit,0.011146,0.011619,0.000833,0.000473,0.000852,9.514834
9,c_09,sigmoid,2,25,"[1, 25, 25, 1]",adam,0.0003,726,new_part_c,,1000,2026,fixed_epoch_limit,0.009580,0.009787,0.000739,0.000207,0.000932,9.552568


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### Pick learning rates using validation folds

Pick one rate per activation/architecture pair from mean validation MSE. Keep training MSE and the paired validation-minus-training gap alongside it. Pick the overall CV winner before viewing any outer test scores. Part c does not assess that test set.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — Part c.

</div>

> Student: Specified mean-validation-MSE selection and comparisons across activation and architecture. LLM-assisted: Codex implemented selection, aggregation, tables and plots on 2 November 2026.


In [4]:
c_valid = c_grid.loc[c_grid["status"] == "fixed_epoch_limit"]
c_picks = c_valid.loc[c_valid.groupby(["activation", "depth", "width"])["cv_mse"].idxmin()].copy()
if len(c_picks) != 18:
    raise RuntimeError("At least one activation/architecture pair has no complete setting")
c_best_id = c_picks.loc[c_picks["cv_mse"].idxmin(), "run_id"]
c_picks["overall_cv_pick"] = c_picks["run_id"] == c_best_id
c_picks.to_csv(c_out / "validation_picks.csv", index=False)
c_display_columns = ["activation", "depth", "width", "parameter_count", "gamma", "train_mse", "cv_mse", "cv_sd", "gap", "gap_sd", "origin"]
display(c_picks[c_display_columns])
print("Overall CV pick:", c_best_id)
print("Fold SD is split spread at fixed seeds. Positive gap alone does not prove overfitting.")
print("Reused Part b times come from the earlier implementation and are not a speed comparison.")

,activation,depth,width,parameter_count,gamma,train_mse,cv_mse,cv_sd,gap,gap_sd,origin
36,leaky_relu,1,25,76,0.0003,0.009680,0.009999,0.000512,0.000320,0.000675,new_part_c
39,leaky_relu,1,50,151,0.0003,0.009627,0.010138,0.000729,0.000511,0.000891,new_part_c
42,leaky_relu,1,100,301,0.0003,0.009685,0.010293,0.000572,0.000608,0.000655,new_part_c
45,leaky_relu,2,25,726,0.0003,0.009551,0.010228,0.000678,0.000677,0.000819,new_part_c
50,leaky_relu,2,50,2701,0.0030,0.009321,0.010247,0.000682,0.000926,0.000842,new_part_c
53,leaky_relu,2,100,10401,0.0030,0.009331,0.010412,0.000442,0.001081,0.000547,new_part_c
18,relu,1,25,76,0.0003,0.009668,0.010000,0.000517,0.000333,0.000675,new_part_c
21,relu,1,50,151,0.0003,0.009628,0.010145,0.000714,0.000518,0.000876,new_part_c
24,relu,1,100,301,0.0003,0.009688,0.010297,0.000569,0.000609,0.000636,new_part_c
27,relu,2,25,726,0.0003,0.009549,0.010259,0.000596,0.000710,0.000750,new_part_c


Overall CV pick: c_09
Fold SD is split spread at fixed seeds. Positive gap alone does not prove overfitting.
Reused Part b times come from the earlier implementation and are not a speed comparison.


### Part d — penalties

### Answers

Use 16 configurations per family and the same five Runge folds at seed 2026. Pick each family by mean validation MSE. The saved selections agree with the historical selections. All network fits keep sigmoid hidden layers, Xavier initialisation, 1000 epochs and batches of 32. Penalties apply to weights only.


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### Reuse the data, folds and preprocessing

Read the saved Part b split. Fit scaling only on each fold’s 640 training rows. Save the search plan before training. Part c informs the fixed architecture list, not a new test-based choice.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — Part d.

</div>

> Student: Specified the weight-only L1/L2 experiment, search ranges, budgets and polynomial baseline comparison. LLM-assisted: Codex implemented Part d configuration, sampling, setup and records on 2 November 2026.


In [ ]:
from pathlib import Path
import json
import hashlib
import importlib.metadata
import time
import runpy
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import loguniform
from sklearn.model_selection import ParameterSampler
from sklearn.linear_model import Lasso
from sklearn.exceptions import ConvergenceWarning
from reused_project1 import (
    scale_data, create_design_matrix, fit_regression, optimize,
    cost_function, predict,
)
from reused_network import (
    create_layers_batched, train_network, feed_forward_batched,
    network_loss, weight_penalty,
)

d_code_dir = Path.cwd() if (Path.cwd() / "reused_network.py").exists() else Path.cwd() / "Code"
d_root = d_code_dir.parent
d_out = d_root / "results/runge_part_d"
d_out.mkdir(exist_ok=True)
d_b_out = d_root / "results/runge_part_b_cv"
d_split = np.load(d_b_out / "data_split.npz")
d_x, d_y = d_split["x"], d_split["y"]
d_train, d_test = d_split["train_pool"], d_split["test"]
d_folds = []
d_fold_data = []
d_scalers = {}
for fold in range(5):
    train_rows, val_rows = d_split[f"fit_{fold}"], d_split[f"validation_{fold}"]
    assert not set(train_rows) & set(val_rows)
    assert not set(train_rows) & set(d_test)
    assert not set(val_rows) & set(d_test)
    X_fit, X_val, y_fit, y_val, mean, sd, y_mean = scale_data(
        d_x[train_rows, None], d_x[val_rows, None], d_y[train_rows], d_y[val_rows],
    )
    d_folds.append((train_rows, val_rows))
    d_fold_data.append({"X_fit": X_fit, "X_val": X_val, "y_fit": y_fit[:, None], "y_val": y_val[:, None], "y_mean": y_mean})
    d_scalers.update({f"x_mean_{fold}": mean, f"x_sd_{fold}": sd, f"y_mean_{fold}": y_mean})
np.savez(d_out / "data_split.npz", **{key: d_split[key] for key in d_split.files})
np.savez(d_out / "fold_preprocessing.npz", **d_scalers)
d_budget = 16
d_seeds = [2026]
d_epochs = 1000
d_spaces = {"hidden": [(25,), (25, 25), (50, 50)], "gamma": loguniform(1e-4, 1e-2), "lmbd": loguniform(1e-7, 1e-2)}
d_samples = list(ParameterSampler(d_spaces, n_iter=d_budget, random_state=42))
d_configs = []
for kind in ["none", "l1", "l2"]:
    for trial, sample in enumerate(d_samples):
        d_configs.append({
            "config_id": f"{kind}_{trial:02d}", "kind": kind, "trial": trial,
            "hidden": [int(width) for width in sample["hidden"]],
            "gamma": float(sample["gamma"]), "lmbd": 0.0 if kind == "none" else float(sample["lmbd"]),
        })
d_config_map = {config["config_id"]: config for config in d_configs}
pd.DataFrame(d_configs).to_csv(d_out / "sampled_configurations.csv", index=False)
(d_out / "configurations.json").write_text(json.dumps(d_configs, indent=2) + "\n")
# Historical five-fold picks, fixed before the present assessment.
d_baselines = json.loads((d_root / "data/polynomial_baselines.json").read_text())
d_poly_configs = [d_baselines["ridge"], d_baselines["lasso"]]
d_settings = {
    "date": "2026-11-02", "optimizer": "adam", "activation": "sigmoid",
    "epochs": d_epochs, "batch_size": 32, "seeds": d_seeds,
    "folds": 5, "fold_seed": 42, "sampling_seed": 42,
    "initialization": "xavier_normal", "biases": "zero and unpenalized", "dtype": "float64",
    "penalty_convention": "full weight-only sum, no extra n/layer/parameter normalization",
    "l1_zero_subgradient": 0, "positive_trials_per_penalty": d_budget,
    "zero_control_trials": d_budget,
    "architectures": [[25], [25, 25], [50, 50]],
    "rate_distribution": "loguniform(1e-4,1e-2)", "penalty_distribution": "loguniform(1e-7,1e-2)",
    "pick_rule": "lowest mean five-fold unpenalized validation MSE at seed 2026",
    "test_use": "after freezing all settings, same historically used Project 1 200-row split",
    "test_seed_rule": "one fixed final fit per network at seed 2026",
    "near_zero_weight_threshold": 1e-3, "convergence": "fixed epoch limit for networks, no early stopping",
    "polynomial_benchmarks": d_poly_configs,
    "lasso_solver": "reused Project 1 Adam/JAX, lr .01, tol 1e-7, max 10000, seed 42",
    "lasso_reference": "sklearn coordinate descent, alpha=lambda/2 for matching full-MSE convention",
    "implementation_sha256": hashlib.sha256((d_code_dir / "reused_network.py").read_bytes()).hexdigest(),
    "dependencies": {name: importlib.metadata.version(name) for name in ["numpy", "scipy", "pandas", "matplotlib", "scikit-learn", "jax", "jaxlib"]},
    "rerun": ".venv/bin/python Code/run_part_d.py from the project root",
}
(d_out / "settings.json").write_text(json.dumps(d_settings, indent=2) + "\n")
d_receipt = json.loads((d_out / "development_checks.json").read_text())
if d_receipt.get("implementation_sha256") != d_settings["implementation_sha256"]:
    runpy.run_path(str(d_code_dir / "check_part_d.py"), run_name="__main__")
    d_receipt = json.loads((d_out / "development_checks.json").read_text())
display(pd.Series(d_receipt, name="Penalty development checks"))
display(pd.DataFrame(d_configs))

#### Shared fold evaluation

Each configuration is fitted at seed 2026. Training minimises the penalised objective. Predictive scores use unpenalised MSE.


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — Part d.

</div>

> Student: Specified the five-fold regularization experiment, initialization and training design. LLM-assisted: Codex implemented the fold evaluator, checkpoints and summary-table helper on 2 November 2026.


In [ ]:
def d_evaluate(config, seed):
    """Fit five folds and save objective and metric results.
    
    Student: Specified the weight-only regularized objectives, five-fold experiment and
    seeded training design.
    
    LLM-assisted: Codex adapted the Part c fold loop and implemented records, checkpoints
    and failure handling.
    """
    hidden = config["hidden"]
    names = ["sigmoid"] * len(hidden) + ["linear"]
    initial = create_layers_batched(1, hidden + [1], seed=seed, initialization="xavier")
    penalty = None if config["kind"] == "none" else config["kind"]
    rows, curves = [], {}
    oof = np.full(len(d_x), np.nan)
    for fold, ((train_rows, val_rows), data) in enumerate(zip(d_folds, d_fold_data)):
        start = time.perf_counter()
        row = {**config, "hidden": str(hidden), "seed": seed, "fold": fold, "epochs": d_epochs, "status": "fixed_epoch_limit", "error": ""}
        try:
            with np.errstate(over="raise", invalid="raise", divide="raise"):
                layers, history = train_network(
                    initial, data["X_fit"], data["y_fit"], names,
                    gamma=config["gamma"], epochs=d_epochs, batch_size=32,
                    method="adam", seed=seed, penalty=penalty, lmbd=config["lmbd"],
                )
                train_mse = network_loss(data["X_fit"], layers, names, data["y_fit"])
                val_prediction = feed_forward_batched(data["X_val"], layers, names)
                val_mse = float(np.mean((val_prediction - data["y_val"]) ** 2))
                if not np.isfinite(val_mse):
                    raise FloatingPointError("Nonfinite validation MSE")
            oof[val_rows] = val_prediction.ravel() + data["y_mean"]
            curves[f"fold_{fold}"] = history
            weights = np.concatenate([W.ravel() for W, _ in layers])
            row.update(
                train_mse=train_mse, validation_mse=val_mse, gap=val_mse-train_mse,
                objective=float(history[-1]), weight_l1=float(np.abs(weights).sum()),
                weight_l2sq=float((weights ** 2).sum()),
                exact_zero_fraction=float(np.mean(weights == 0)),
                near_zero_fraction=float(np.mean(np.abs(weights) < 1e-3)),
            )
            np.savez(
                d_out / f"{config['config_id']}_seed_{seed}_fold_{fold}_weights.npz",
                **{f"W_{i}": W for i, (W, b) in enumerate(layers)},
                **{f"b_{i}": b for i, (W, b) in enumerate(layers)},
            )
        except FloatingPointError as error:
            # keep failed trials, without retrying beyond the equal budgets.
            row.update(status="failed", error=str(error), train_mse=np.nan, validation_mse=np.nan, gap=np.nan, objective=np.nan)
        row["seconds"] = time.perf_counter() - start
        rows.append(row)
    np.savez(d_out / f"{config['config_id']}_seed_{seed}_objective_curves.npz", **curves)
    np.savez(d_out / f"{config['config_id']}_seed_{seed}_oof.npz", prediction=oof[d_train])
    print(config["config_id"], "seed", seed, "five folds", flush=True)
    return rows


def d_summarize(rows):
    """Summarize one five-fold setting and seed.
    
    LLM-assisted: Codex implemented the result aggregation and table helper for the
    specified selection rule.
    """
    frame = pd.DataFrame(rows)
    result = []
    for (config_id, seed), group in frame.groupby(["config_id", "seed"]):
        config = d_config_map[config_id]
        complete = len(group) == 5 and (group["status"] == "fixed_epoch_limit").all()
        result.append({
            **config, "hidden": str(config["hidden"]), "seed": int(seed),
            "status": "complete" if complete else "failed",
            "train_mse": group["train_mse"].mean() if complete else np.nan,
            "cv_mse": group["validation_mse"].mean() if complete else np.nan,
            "cv_fold_sd": group["validation_mse"].std(ddof=1) if complete else np.nan,
            "gap": group["gap"].mean() if complete else np.nan,
            "gap_fold_sd": group["gap"].std(ddof=1) if complete else np.nan,
            "seconds": group["seconds"].sum(),
        })
    return pd.DataFrame(result)

#### Five-fold random search

Evaluate the 16 fixed draws for each family on the shared folds.


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — Part d.

</div>

> Student: Specified the fixed five-fold search over the penalty configurations. LLM-assisted: Codex implemented the search orchestration, saved records and table displays on 2 November 2026.


In [ ]:
d_fold_rows = []
for config in d_configs:
    d_fold_rows.extend(d_evaluate(config, d_seeds[0]))
    pd.DataFrame(d_fold_rows).to_csv(d_out / "fold_scores.csv", index=False)
    d_summarize(d_fold_rows).to_csv(d_out / "initial_search.csv", index=False)
d_initial = d_summarize(d_fold_rows)
display(d_initial.sort_values(["kind", "cv_mse"]))


#### Validation picks

Pick the lowest mean five-fold validation MSE in each family before final assessment.


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — Part d.

</div>

> Student: Specified selection by lowest mean five-fold validation MSE and freezing before test evaluation. LLM-assisted: Codex implemented selection, validation, frozen settings and displays on 2 November 2026.


In [ ]:
d_selected = d_initial.loc[
    d_initial.loc[d_initial.status == "complete"].groupby("kind").cv_mse.idxmin()
].copy()
if set(d_selected.kind) != {"none", "l1", "l2"}:
    raise RuntimeError("Each penalty family needs a complete validation pick")
d_selected.to_csv(d_out / "selected_configurations.csv", index=False)
d_initial.to_csv(d_out / "seed_scores.csv", index=False)
(d_out / "frozen_selection.json").write_text(json.dumps({
    "selected": [d_config_map[cid] for cid in d_selected.config_id],
    "seeds": [2026], "basis": "lowest mean five-fold validation MSE at seed 2026",
    "test_scores_used": False,
}, indent=2) + "\n")
display(d_selected)


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

#### Project 1 Ridge and Lasso on the same folds

Keep the historical Project 1 CV-picked degree and strength fixed. Reuse its polynomial features, scaling, Ridge solver and Lasso Adam/JAX fitter. Compare Ridge with an augmented least-squares solution and Lasso with coordinate descent using alpha = lambda/2, which matches the full-MSE objective. Report iterations and failures explicitly.

</div>


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — Part d.

</div>

> Student: Specified matched-fold Ridge/Lasso baseline objectives and comparison design. LLM-assisted: Codex implemented baseline fitting, independent checks, convergence handling and output tables on 2 November 2026.


In [5]:
d_poly_folds = []
d_poly_oof = {}
for config in d_poly_configs:
    oof = np.full(len(d_x), np.nan)
    for fold, (train_rows, val_rows) in enumerate(d_folds):
        X_fit = create_design_matrix(d_x[train_rows], config["degree"])
        X_val = create_design_matrix(d_x[val_rows], config["degree"])
        X_fit, X_val, y_fit, _, mean, sd, y_mean = scale_data(X_fit, X_val, d_y[train_rows])
        if config["model"] == "ridge":
            theta = fit_regression(X_fit, y_fit, lambda_=config["lmbd"])
            augmented = np.vstack([X_fit, np.sqrt(len(train_rows) * config["lmbd"]) * np.eye(config["degree"])])
            target = np.r_[y_fit, np.zeros(config["degree"])]
            reference = np.linalg.lstsq(augmented, target, rcond=None)[0]
            iterations, converged, reference_dual_gap = 1, True, np.nan
        else:
            theta, iterations, history, converged = optimize(
                X_fit, y_fit, lambda_=config["lmbd"], regularization="lasso",
                gradient_source="automatic", method="adam", learning_rate=.01,
                tolerance=1e-7, max_iterations=10000, seed=42,
            )
            with warnings.catch_warnings():
                warnings.simplefilter("error", ConvergenceWarning)
                reference_model = Lasso(alpha=config["lmbd"] / 2, fit_intercept=False, tol=1e-12, max_iter=100000).fit(X_fit, y_fit)
            reference = reference_model.coef_
            reference_dual_gap = float(reference_model.dual_gap_)
        train_prediction = predict(X_fit, theta, y_mean)
        val_prediction = predict(X_val, theta, y_mean)
        oof[val_rows] = val_prediction
        penalty_name = "ridge" if config["model"] == "ridge" else "lasso"
        objective = float(cost_function(theta, X_fit, y_fit, config["lmbd"], penalty_name))
        reference_objective = float(cost_function(reference, X_fit, y_fit, config["lmbd"], penalty_name))
        d_poly_folds.append({
            **config, "fold": fold, "train_mse": float(np.mean((d_y[train_rows] - train_prediction) ** 2)),
            "validation_mse": float(np.mean((d_y[val_rows] - val_prediction) ** 2)),
            "iterations": iterations, "converged": bool(converged),
            "objective_gap_to_reference": objective-reference_objective,
            "max_validation_prediction_difference": float(np.max(np.abs(X_val @ (theta-reference)))),
            "reference_dual_gap": reference_dual_gap,
        })
        np.savez(d_out / f"{config['model']}_fold_{fold}_weights.npz", theta=theta, x_mean=mean, x_sd=sd, y_mean=y_mean, reference_theta=reference)
        print(config["model"], "fold", fold, "converged", converged, "iterations", iterations, flush=True)
    d_poly_oof[config["model"]] = oof[d_train]
d_poly_folds = pd.DataFrame(d_poly_folds)
d_poly_folds.to_csv(d_out / "polynomial_fold_scores.csv", index=False)
np.savez(d_out / "polynomial_oof.npz", **d_poly_oof)
d_poly_cv = d_poly_folds.groupby("model").agg(
    cv_mse=("validation_mse", "mean"), cv_fold_sd=("validation_mse", "std"),
    train_mse=("train_mse", "mean"), all_converged=("converged", "all"),
).reset_index()
d_poly_cv.to_csv(d_out / "polynomial_cv.csv", index=False)
if not d_poly_cv["all_converged"].all():
    raise RuntimeError("A polynomial benchmark reached its iteration limit. Inspect diagnostics before test scoring.")
display(d_poly_folds)
display(d_poly_cv)

ridge fold 0 converged True iterations 1


ridge fold 1 converged True iterations 1


ridge fold 2 converged True iterations 1


ridge fold 3 converged True iterations 1


ridge fold 4 converged True iterations 1


lasso fold 0 converged True iterations 1693


... (2 outputs left out)

lasso fold 3 converged True iterations 1694


lasso fold 4 converged True iterations 1736


,model,degree,lmbd,historical_cv_mse,fold,train_mse,validation_mse,iterations,converged,objective_gap_to_reference,max_validation_prediction_difference,reference_dual_gap
0,ridge,14,0.00001,0.012994,0,0.012076,0.013459,1,True,-1.734723e-18,2.032272e-12,NaN
1,ridge,14,0.00001,0.012994,1,0.012431,0.012203,1,True,3.469447e-18,4.281782e-13,NaN
2,ridge,14,0.00001,0.012994,2,0.012258,0.012702,1,True,-5.204170e-18,5.749352e-12,NaN
3,ridge,14,0.00001,0.012994,3,0.012462,0.012240,1,True,5.204170e-18,2.980296e-12,NaN
4,ridge,14,0.00001,0.012994,4,0.012258,0.012323,1,True,-1.734723e-18,9.994598e-13,NaN
5,lasso,6,0.00010,0.018024,0,0.017220,0.018982,1693,True,7.310125e-14,9.992322e-07,8.868600e-14
6,lasso,6,0.00010,0.018024,1,0.017432,0.018136,1712,True,7.986320e-14,6.442295e-07,9.189038e-14
7,lasso,6,0.00010,0.018024,2,0.017785,0.016661,1801,True,9.222484e-14,8.052895e-07,9.219431e-14
8,lasso,6,0.00010,0.018024,3,0.017681,0.017219,1694,True,7.533904e-14,3.781715e-07,8.978235e-14
9,lasso,6,0.00010,0.018024,4,0.017455,0.018004,1736,True,8.276366e-14,8.280432e-07,9.556939e-14


,model,cv_mse,cv_fold_sd,train_mse,all_converged
0,lasso,0.017800,0.000893,0.017515,True
1,ridge,0.012585,0.000527,0.012297,True


#### Final refits

Refit the fixed network settings at seed 2026 and compare the fixed polynomial baselines on the same test rows.


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**PROJECT CODE** — Part d.

</div>

> Student: Specified full-development refits and final test comparison of frozen settings. LLM-assisted: Codex implemented model fitting, scoring, plots and saved outputs on 2 November 2026.


In [6]:
assert (d_out / "frozen_selection.json").exists()
d_X_train, d_X_test, d_y_train, _, d_mean, d_sd, d_y_mean = scale_data(d_x[d_train, None], d_x[d_test, None], d_y[d_train])
d_y_train = d_y_train[:, None]
np.savez(d_out / "final_preprocessing.npz", x_mean=d_mean, x_sd=d_sd, y_mean=d_y_mean)
d_test_records = []
d_test_predictions = {}
d_refit_curves = {}
for _, selected in d_selected.iterrows():
    config = d_config_map[selected.config_id]
    names = ["sigmoid"] * len(config["hidden"]) + ["linear"]
    penalty = None if config["kind"] == "none" else config["kind"]
    for seed in d_seeds:
        initial = create_layers_batched(1, config["hidden"] + [1], seed=seed, initialization="xavier")
        layers, objective_curve = train_network(
            initial, d_X_train, d_y_train, names,
            gamma=config["gamma"], epochs=d_epochs, batch_size=32, method="adam",
            seed=seed, penalty=penalty, lmbd=config["lmbd"],
        )
        prediction = feed_forward_batched(d_X_test, layers, names).ravel() + d_y_mean
        key = f"{config['kind']}_seed_{seed}"
        d_test_predictions[key] = prediction
        d_refit_curves[key] = objective_curve
        d_test_records.append({
            "model": config["kind"], "config_id": config["config_id"], "seed": seed,
            "hidden": str(config["hidden"]), "gamma": config["gamma"], "lmbd": config["lmbd"],
            "train_mse": network_loss(d_X_train, layers, names, d_y_train),
            "test_mse": float(np.mean((d_y[d_test] - prediction) ** 2)),
            "training_objective": float(objective_curve[-1]),
            "weight_l1": weight_penalty(layers, "l1", 1), "weight_l2sq": weight_penalty(layers, "l2", 1),
        })
        np.savez(d_out / f"final_{key}_weights.npz", **{f"W_{i}": W for i, (W,b) in enumerate(layers)}, **{f"b_{i}": b for i, (W,b) in enumerate(layers)})
for config in d_poly_configs:
    X_train = create_design_matrix(d_x[d_train], config["degree"])
    X_test = create_design_matrix(d_x[d_test], config["degree"])
    X_train, X_test, y_train, _, mean, sd, y_mean = scale_data(X_train, X_test, d_y[d_train])
    if config["model"] == "ridge":
        theta = fit_regression(X_train, y_train, lambda_=config["lmbd"])
        iterations, converged = 1, True
    else:
        theta, iterations, history, converged = optimize(X_train, y_train, lambda_=config["lmbd"], regularization="lasso", gradient_source="automatic", method="adam", learning_rate=.01, tolerance=1e-7, max_iterations=10000, seed=42)
    if not converged:
        raise RuntimeError(f"Final {config['model']} fit reached the iteration limit")
    prediction = predict(X_test, theta, y_mean)
    d_test_predictions[config["model"]] = prediction
    d_test_records.append({
        "model": config["model"], "config_id": config["model"], "seed": 42,
        "hidden": "", "gamma": np.nan, "lmbd": config["lmbd"], "degree": config["degree"],
        "train_mse": float(np.mean((d_y[d_train] - predict(X_train, theta, y_mean)) ** 2)),
        "test_mse": float(np.mean((d_y[d_test] - prediction) ** 2)),
        "iterations": iterations, "converged": converged,
    })
    np.savez(d_out / f"final_{config['model']}_weights.npz", theta=theta, x_mean=mean, x_sd=sd, y_mean=y_mean)
d_test_records = pd.DataFrame(d_test_records)
d_test_records.to_csv(d_out / "test_scores.csv", index=False)
np.savez(d_out / "test_predictions.npz", **d_test_predictions)
np.savez(d_out / "final_objective_curves.npz", **d_refit_curves)
d_test_summary = d_test_records.groupby("model").agg(
    mean_train_mse=("train_mse", "mean"), mean_test_mse=("test_mse", "mean"),
    test_seed_sd=("test_mse", "std"), fits=("test_mse", "count"),
).reset_index()
d_test_summary.to_csv(d_out / "test_summary.csv", index=False)
display(d_selected)
display(d_test_records)
display(d_test_summary)
print("No test-based tuning or seed picking follows. These are reused Project 1 test rows.")

,config_id,kind,trial,hidden,gamma,lmbd,status,seeds,train_mse,cv_mse,cv_seed_sd,mean_cv_fold_sd,gap
2,l1_04,l1,4,"[25, 25]",0.008706,0.000001,complete,3,0.009658,0.009937,0.000033,0.000854,0.000278
4,l2_04,l2,4,"[25, 25]",0.008706,0.000001,complete,3,0.009671,0.009934,0.000054,0.000807,0.000263
0,none_03,none,3,"[50, 50]",0.000465,0.000000,complete,3,0.009675,0.009947,0.000019,0.000771,0.000272


,model,config_id,seed,hidden,gamma,lmbd,train_mse,test_mse,training_objective,weight_l1,weight_l2sq,degree,iterations,converged
0,l1,l1_04,2026,"[25, 25]",0.008706,0.000001,0.009857,0.010931,0.010034,153.706084,325.439585,NaN,NaN,NaN
1,l1,l1_04,2027,"[25, 25]",0.008706,0.000001,0.009658,0.010558,0.009776,101.906557,256.452181,NaN,NaN,NaN
2,l1,l1_04,2028,"[25, 25]",0.008706,0.000001,0.010054,0.011112,0.010175,104.996424,366.182408,NaN,NaN,NaN
3,l2,l2_04,2026,"[25, 25]",0.008706,0.000001,0.009757,0.010805,0.009979,143.972819,192.198430,NaN,NaN,NaN
4,l2,l2_04,2027,"[25, 25]",0.008706,0.000001,0.009653,0.010583,0.009879,96.758346,195.653575,NaN,NaN,NaN
5,l2,l2_04,2028,"[25, 25]",0.008706,0.000001,0.010153,0.011207,0.010411,101.750137,223.349946,NaN,NaN,NaN
6,none,none_03,2026,"[50, 50]",0.000465,0.000000,0.009887,0.010810,0.009887,425.778583,219.036813,NaN,NaN,NaN
7,none,none_03,2027,"[50, 50]",0.000465,0.000000,0.009590,0.010482,0.009590,449.912852,231.253675,NaN,NaN,NaN
8,none,none_03,2028,"[50, 50]",0.000465,0.000000,0.010117,0.011265,0.010117,463.136052,266.398800,NaN,NaN,NaN
9,ridge,ridge,42,,NaN,0.000010,0.012324,0.014437,NaN,NaN,NaN,14.0,1.0,True


,model,mean_train_mse,mean_test_mse,test_seed_sd,fits
0,l1,0.009856,0.010867,0.000283,3
1,l2,0.009855,0.010865,0.000316,3
2,lasso,0.017546,0.018774,NaN,1
3,none,0.009865,0.010852,0.000393,3
4,ridge,0.012324,0.014437,NaN,1


No test-based tuning or seed picking follows. These are reused Project 1 test rows.




> LLM-assisted: Codex implemented this view of saved classification results. Training is in part_e_classification.py.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from part_e_classification import OUT as E_OUT
display(pd.Series(json.loads((E_OUT / "design.json").read_text()), name="Fixed classification design"))


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**Training and checks.** Each shuffled minibatch runs the existing forward pass, mean CE, manual backward sweep and reused Adam update. Only the grouped monitoring CE controls early stopping. Development checks compare every weight/bias gradient to JAX automatic differentiation across zero to three hidden layers, all three activations and all penalty choices. Automatic differentiation is used for checking, not in the own-network training runs.

</div>




> LLM-assisted: Codex implemented the saved split-coverage table display.


In [ ]:
display(pd.read_csv(E_OUT / "fold_class_coverage.csv"))


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**Random search and picking.** ParameterSampler sampled 12 settings, giving 60 fold fits. This is the user-requested runtime reduction. The earlier incomplete full-data run is kept separately as pilot evidence, not mixed into these results. Hidden depth 1/2/3, equal width 32/64/128, sigmoid/ReLU/Leaky ReLU, Adam rates log-uniform from 1e-4 to 1e-2, and none/L1/L2 with positive strengths log-uniform from 1e-7 to 1e-2. Batch size is 128. The reduced-fold pilot and saved runtime decision precede this search. The 24,000-row sample keeps natural class frequencies and uses seed 2026. Unselected development rows enter only the fixed final refits.

Pick the highest mean outer validation macro-F1. Also keep accuracy, unpenalized CE, sample fold SD, runtime and ceiling flags. Macro-F1 is the unweighted mean of the 24 per-class F1 scores, with zero division set to zero. Fold spread is not a confidence interval. Reduced data and the lower ceiling limit the search, so this is a compute-limited study rather than a fully tuned classifier. [Bergstra and Bengio (2012)](https://jmlr.org/papers/v13/bergstra12a.html) supports bounded random search as a baseline, without claiming a matched grid-search comparison here.

</div>




> LLM-assisted: Codex implemented the saved validation-score table display.


In [ ]:
display(pd.read_csv(E_OUT / "search_summary.csv").sort_values("macro_f1", ascending=False))


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**Comparisons and final refits.** Logistic regression is the own softmax network with zero hidden layers, tuning three unpenalized Adam rates on the same folds. The sklearn FFNN matches the best sampled no-penalty sigmoid/ReLU manual configuration. It keeps sklearn's own initialization, shuffle RNG and Adam implementation, with the same data, architecture, activation, rate, precision, batch size and grouped CE stopping. It is compared with its exact manual match as well as the overall manual pick. The model families do not have equal search budgets.

Before test access, freeze each model and its full-development epoch budget: the median of best monitoring epochs times fold-fit rows, divided by 159,053 and rounded up to whole epochs. Refit on all development rows for that budget, then assess the three fixed models on the same reserved test rows.

</div>




> LLM-assisted: Codex implemented the saved model-comparison table display.


In [ ]:
display(pd.read_csv(E_OUT / "model_comparison.csv"))


<div class="proj2-project" style="color: #F5F5F5; background-color: #252733; border-left: 3px solid #F5F5F5; padding: 0.55em 0.9em; margin: 0.4em 0; border-radius: 4px;">

**Learning-curve key.** Blue is online pre-update minibatch training CE, orange is grouped monitoring CE, and black dots mark restored epochs. Five fold curves are shown for each model. Online batch CE uses changing weights and is not a fixed-weight epoch-end training score. The comparison table also reports training scores at the restored weights.

</div>




> LLM-assisted: Codex implemented the saved final-test and per-class table displays.


In [ ]:
display(pd.read_csv(E_OUT / "test_scores.csv"))
display(pd.read_csv(E_OUT / "selected_per_class.csv"))
